In [15]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [16]:
load_dotenv()

True

In [17]:
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

In [18]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [19]:
def create_outline(state: BlogState) -> BlogState:
    # fetch title 
    title = state['title']

    # generate outline
    prompt = f'Write a detailed outline for the blog on the topic {title}'
    outline = model.invoke(prompt).content

    state['outline'] = outline
    return state

In [20]:
def create_blog(state: BlogState) -> BlogState:
    # fetch title outline
    title = state['title']
    outline = state['outline']

    # generate blog
    prompt = f'Write a detailed blog on title - {title} using following outline - {outline}'
    content = model.invoke(prompt).content

    state['content'] = content
    return state

In [21]:
graph = StateGraph(BlogState)

# add node
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

# add edge
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

# compile graph
workflow = graph.compile()

In [22]:
initial_state = {'title': 'Rise of Crime in USA'}
final_state = workflow.invoke(initial_state)
print(final_state)

{'title': 'Rise of Crime in USA', 'outline': 'This is a comprehensive and sensitive topic, so the blog outline should be well-structured, data-driven, and offer a balanced perspective.\n\n---\n\n## Blog Outline: The Rise of Crime in the USA: Understanding the Trends, Causes, and Solutions\n\n**Target Audience:** General public, concerned citizens, policymakers, students of sociology/criminology.\n**Tone:** Informative, analytical, balanced, concerned yet solution-oriented.\n**Keywords:** Crime rates USA, violent crime, property crime, criminal justice, public safety, crime trends, social factors, policing, community solutions.\n**Meta Description:** Explore the recent rise in crime across the USA, delving into the data, complex underlying causes, and potential strategies for creating safer communities.\n\n---\n\n### I. Catchy Title Options:\n*   The Alarming Surge: Unpacking the Rise of Crime in America\n*   Beyond the Headlines: A Deep Dive into America\'s Rising Crime Rates\n*   From

In [23]:
print(final_state['content'])

## Beyond the Headlines: A Deep Dive into America's Rising Crime Rates

From bustling city centers to quiet suburban streets, a palpable sense of unease about public safety has permeated conversations across the United States. Recent headlines and personal experiences suggest a troubling trend: crime rates are on the rise. This uptick is particularly noteworthy given the significant decline in crime the nation experienced from the 1990s through the late 2010s. The post-pandemic acceleration of these trends has only amplified concerns, prompting a nationwide re-evaluation of public safety.

This blog aims to move beyond the headlines, exploring the data that confirms the recent surge in crime, delving into its multifaceted causes, examining its far-reaching impacts on individuals and communities, and discussing potential solutions for a safer future. It's a complex issue with no single cause or easy fix, demanding a nuanced understanding and collaborative action.

### Confirming the Tre